Binary MLP classifier for MNIST even/odd classification, trained using maximum likelihood estimation.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


train_df = pd.read_csv("mnist_train.csv", header=None)
test_df = pd.read_csv("mnist_test.csv", header=None)

# x - array of pixel intensities, y - array of labels

X_train = train_df.iloc[:, 1:].values
y_train = train_df.iloc[:, 0].values

X_test = test_df.iloc[:, 1:].values
y_test = test_df.iloc[:, 0].values



# Normalize intesities 

X_train = X_train / 255.0
X_test = X_test / 255.0



# Convert labels to parity 

y_train_binary = y_train % 2
y_test_binary = y_test % 2

In [2]:
# Number of hidden neurons
H = 64

np.random.seed(0)

# Input -> hidden
# Create a matrix of weights for the input to hidden layer, initialized with small random values
W1 = np.random.randn(784, H) * 0.01
# Create a vector of biases for the hidden layer, initialized with zeros
b1 = np.zeros((1, H)) 


# Hidden -> output
# Create a matrix of weights for the hidden to output layer, initialized with small random values
W2 = np.random.randn(H, 1) * 0.01
# Create a scalar bias for the output layer, initialized with zero
b2 = 0.0 

In [4]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# reshape labels to column vectors
y_train_binary = y_train_binary.reshape(-1, 1)
y_test_binary = y_test_binary.reshape(-1, 1)

In [ ]:
# Forward pass function to compute activations for both layers

def forward(X, W1, b1, W2, b2):
    # Compute activations for the hidden layer
    Z1 = X @ W1 + b1 
    A1 = sigmoid(Z1)

    # Compute activations for the output layer
    Z2 = A1 @ W2 + b2
    A2 = sigmoid(Z2)

    return Z1, A1, Z2, A2

In [ ]:
# Binary cross-entropy loss function to compute the loss between true labels and predicted probabilities

def binary_cross_entropy(y, y_hat):
    eps = 1e-10
    y_hat = np.clip(y_hat, eps, 1 - eps)
    return -np.mean(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))

In [ ]:
# Training loop: estimate errors, do backpropagation, update weights.

lerning_rate = 0.1
loops = 1000

m = X_train.shape[0]

losses = []

for loop_number in range(loops):
    # forward pass
    Z1, A1, Z2, y_hat = forward(X_train, W1, b1, W2, b2)

    # loss
    loss = binary_cross_entropy(y_train_binary, y_hat)
    losses.append(loss)

    # backpropagation
    dZ2 = y_hat - y_train_binary
    dW2 = A1.T @ dZ2 / m
    db2 = np.mean(dZ2)

    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * A1 * (1 - A1)
    dW1 = X_train.T @ dZ1 / m
    db1 = np.mean(dZ1, axis=0, keepdims=True)

    # update weights
    W1 -= lerning_rate * dW1
    b1 -= lerning_rate * db1
    W2 -= lerning_rate * dW2
    b2 -= lerning_rate * db2

    if loop_number % 100 == 0:
        print(f"Epoch {loop_number}, Loss: {loss:.4f}")